# 01 — Substrate

The non-agent machinery every role will depend on: the finding store, the work queue, and the budget governor. No LLM calls in this notebook — the point is to see the constitution's structural guarantees (not prompt instructions) hold on their own, before any agent touches them.

Each section below demonstrates one constitution principle live. The same assertions live in `tests/test_finding_store.py` if you'd rather run them as a suite (`pytest tests/ -v`) — this notebook is the same proofs, run interactively so you can see the state change at each step.

In [ ]:
import tempfile
import time
import threading
from pathlib import Path

from foundry.substrate.db import connect
from foundry.substrate.finding_store import Citation, FindingStore, fingerprint
from foundry.substrate.work_queue import WorkQueue
from foundry.substrate.budget import BudgetCaps, BudgetGovernor

db_path = Path(tempfile.mkdtemp()) / "foundry.sqlite3"
print(f"Using scratch database: {db_path}")

## Constitution VIII — Fingerprints Are Stable Under Edit

A finding's identity is `(normalized_path, symbol, vulnerability_class)` — never a line number or snippet. Re-queueing the same candidate after an unrelated edit (simulated here by changing only the description) must not create a duplicate.

In [ ]:
conn = connect(db_path)
store = FindingStore(conn)

id1, fp1, was_new1 = store.queue_candidate(
    normalized_path="data/toy_target/vulnerable_app.py",
    symbol="get_user_by_name",
    vulnerability_class="sql-injection",
    description="Detected on first sweep",
    technique="codeguard-rule:input-validation-injection",
)
print(f"First queue:  id={id1} fingerprint={fp1} was_new={was_new1}")

# Simulate a re-run after the function moved a few lines -- only the
# description text differs, identity fields are unchanged.
id2, fp2, was_new2 = store.queue_candidate(
    normalized_path="data/toy_target/vulnerable_app.py",
    symbol="get_user_by_name",
    vulnerability_class="sql-injection",
    description="Detected on second sweep, function now 4 lines lower",
    technique="codeguard-rule:input-validation-injection",
)
print(f"Second queue: id={id2} fingerprint={fp2} was_new={was_new2}")
assert id1 == id2 and not was_new2, "should have deduplicated, not re-filed"

## Constitution I — Evidence Over Assertion

`assign_verdict()` will not accept `true-positive` unless every citation resolves against a resolver. Here the resolver is a small fake symbol table standing in for the real Indexer (which lands in notebook 02) — the mechanism is identical either way. First: a clean pass. Then: a deliberately fabricated citation, to watch it get demoted rather than silently accepted.

In [ ]:
known_symbols = {"get_user_by_name", "users_endpoint", "read_uploaded_file", "files_endpoint"}

def fake_resolver(c: Citation) -> bool:
    return c.symbol in known_symbols

clean_citations = [
    Citation("data/toy_target/vulnerable_app.py", "users_endpoint", "reachability"),
    Citation("data/toy_target/vulnerable_app.py", "get_user_by_name", "impact"),
]
verdict = store.assign_verdict(id1, "true-positive", clean_citations, "clean investigation", fake_resolver)
print(f"Clean citations -> verdict: {verdict}")
assert verdict == "true-positive"

In [ ]:
id3, _, _ = store.queue_candidate(
    normalized_path="data/toy_target/vulnerable_app.py",
    symbol="read_uploaded_file",
    vulnerability_class="path-traversal",
    description="candidate",
    technique="exploratory",
)

fabricated_citations = [
    Citation("data/toy_target/vulnerable_app.py", "sanitize_path_properly", "reachability"),  # does not exist
    Citation("data/toy_target/vulnerable_app.py", "read_uploaded_file", "impact"),
]
verdict = store.assign_verdict(id3, "true-positive", fabricated_citations, "confident but wrong", fake_resolver)
print(f"Fabricated citation -> verdict: {verdict}")
assert verdict == "needs-review", "should have been demoted, not accepted as true-positive"

row = store.get(id3)
print("\nRecorded investigation report:\n", row["investigation_report"])

## Constitution IV — Claims Are Atomic And Mortal

Enqueue several tasks, then race multiple worker threads (each with its own SQLite connection, simulating separate agent processes) to claim them. No task should ever be claimed by more than one worker, and none should be lost.

In [ ]:
queue = WorkQueue(conn, lease_seconds=60)
n_tasks, n_workers = 25, 8
task_ids = {queue.enqueue("index_function", {"i": i}) for i in range(n_tasks)}

claimed_by: dict[int, list[str]] = {}
lock = threading.Lock()

def worker(worker_id: str) -> None:
    wconn = connect(db_path)
    wqueue = WorkQueue(wconn, lease_seconds=60)
    while True:
        task = wqueue.claim_next(worker_id, task_type="index_function")
        if task is None:
            break
        with lock:
            claimed_by.setdefault(task.id, []).append(worker_id)
        wqueue.release(task.id, worker_id, status="done")
    wconn.close()

threads = [threading.Thread(target=worker, args=(f"worker-{i}",)) for i in range(n_workers)]
[t.start() for t in threads]
[t.join(timeout=30) for t in threads]

assert set(claimed_by.keys()) == task_ids, "every task should be claimed exactly once, none lost"
assert all(len(w) == 1 for w in claimed_by.values()), "no task should ever be double-claimed"
print(f"{len(task_ids)} tasks, {n_workers} racing workers -> every task claimed exactly once. No duplicates, none lost.")

## Constitution III — Liveness By Heartbeat, Never By Clock

A claim is only reclaimable once its lease has expired -- never on a fixed wall-clock timeout. Below: a task claimed under a zero-second lease becomes immediately reclaimable; a task claimed under a normal lease does not.

In [ ]:
stale_queue = WorkQueue(conn, lease_seconds=0)  # lease expires immediately
stale_task_id = stale_queue.enqueue("probe", {})
claimed = stale_queue.claim_next("agent-a", task_type="probe")
time.sleep(1.1)

reclaimer_conn = connect(db_path)
reclaimer_queue = WorkQueue(reclaimer_conn, lease_seconds=60)
reclaimed = reclaimer_queue.claim_next("agent-b", task_type="probe")
print(f"Stale claim reclaimed by agent-b: {reclaimed is not None and reclaimed.id == stale_task_id}")
assert reclaimed is not None and reclaimed.id == stale_task_id

fresh_queue = WorkQueue(conn, lease_seconds=60)
fresh_task_id = fresh_queue.enqueue("probe", {})
fresh_queue.claim_next("agent-c", task_type="probe")
stolen = reclaimer_queue.claim_next("agent-d", task_type="probe")
print(f"Fresh claim stolen while agent-c still heartbeating: {stolen is not None and stolen.id == fresh_task_id}")
assert not (stolen is not None and stolen.id == fresh_task_id), "a live claim should not be reclaimable"

## Constitution VI — Coverage Before Yield

`should_stop()` is a conjunction: low yield alone never halts the fleet while coverage is incomplete. Three scenarios: incomplete coverage with zero yield (must not stop), complete coverage with low yield (must stop), complete coverage with healthy yield (must not stop).

In [ ]:
gov = BudgetGovernor(conn, BudgetCaps(yield_threshold=0.5))
gov.record_spend(100.0, "detector sweep so far")

stop, reason = gov.should_stop(coverage_complete=False)
print(f"Coverage incomplete, zero yield -> stop={stop} ({reason})")
assert stop is False, "must not stop on yield alone while coverage is incomplete"

stop, reason = gov.should_stop(coverage_complete=True)
print(f"Coverage complete, zero yield -> stop={stop} ({reason})")
assert stop is True

## Recap

Every principle above held under a live workload, with concurrent threads standing in for a real multi-agent fleet -- no LLM was involved anywhere in this notebook. `notebooks/02_indexer.ipynb` (not built yet) is where the first real OpenAI-backed agent lands, reading `data/toy_target/vulnerable_app.py` and exposing it to the rest of the fleet through the query interface spec.md §5.2 requires.